# Re-run: the prediction floor (RQ2)

Each `floor` record holds, for one model, corpus and training size, the oracle's effect-minus-null difference O,
G_N with its interval and G_E - G_N with its interval, as the benchmark's scorer wrote them from five synthetic
twin pairs. This notebook re-applies the pass rule (G_N's interval not above zero; G_E - G_N's interval above
zero with a point estimate of at least O/2), checks it against the scorer's reading, and checks Table 4's
printed values and the bars O/2. It needs only the standard library.

The floors of PFA, BKT and the running proportion were run outside the registered stage lists. Their readings are
checked here like every other, but the article reports them as descriptive only, not as passes; the floors that
count are the six deep models'.

In [1]:
# Finding the published data. This cell is the same in every notebook of this report.
import gzip, json, os, sys
from pathlib import Path

# Importing from the release's code/ folder must not leave __pycache__ folders in it.
sys.dont_write_bytecode = True

RELEASE_ID = "knowledge-tracing-benchmark-2026"   # set when the report is built
DATA_NAME = 'per-case-outputs.jsonl'     # this release's per-case file, set when the report is built (None: it has none)


def find_release_file(name=None):
    """Look beside this notebook, in RELEASE_DIR, in the folders above, and in a repository checkout."""
    name = name or DATA_NAME
    if not name:
        raise FileNotFoundError(f"Release {RELEASE_ID} publishes no per-case file; this notebook reads its "
                                f"code package (code_dir).")
    here = Path.cwd()
    places = [Path(os.environ["RELEASE_DIR"])] if os.environ.get("RELEASE_DIR") else []
    places += [here, *here.parents]
    places += [p / "research/wiki/research/releases" / RELEASE_ID for p in [here, *here.parents]]
    for d in places:
        if (d / name).is_file():
            return d / name
        gz = d / (name + ".gz")
        if gz.is_file():
            # The report offers the data file gzipped; unpack it once beside the download, so a notebook that
            # opens the file as text reads the same bytes either way.
            try:
                (d / name).write_bytes(gzip.decompress(gz.read_bytes()))
                return d / name
            except OSError:
                return gz
    raise FileNotFoundError(
        f"Could not find {name} (or {name}.gz). Put it in the same folder as this notebook: download it "
        f"from the report ('Download the data file', beside each notebook) or take it from the release "
        f"directory. Or set the environment variable RELEASE_DIR to the folder that holds it.")


def per_case(runs=None):
    """Yield the per-case records of the published file, optionally only for the named runs."""
    path = find_release_file()
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            if r.get("record") == "case" and (runs is None or r["run"] in runs):
                yield r


def is_this_release(d):
    """True when folder d holds this release, not another one's code/ found higher up: d is named after the
    release, or its code-manifest.txt or a release file (*.yaml) names it."""
    if d.name == RELEASE_ID:
        return True
    texts = [d / "code-manifest.txt", *sorted(d.glob("*.yaml"))]
    return any(f.is_file() and RELEASE_ID in f.read_text(encoding="utf-8", errors="ignore") for f in texts)


def code_dir():
    """The release's code package (its code/ folder), looked for only by a notebook that reads it: in RELEASE_DIR, beside this notebook or in a folder
    above it (as when the notebook sits in the unzipped release's notebooks/ folder), in a folder named
    after the release there, or in a repository checkout."""
    here = Path.cwd()
    places = [Path(os.environ["RELEASE_DIR"])] if os.environ.get("RELEASE_DIR") else []
    for p in [here, *here.parents]:
        places += [p, p / RELEASE_ID, p / "research/wiki/research/releases" / RELEASE_ID]
    for d in places:
        if (d / "code" / "SHA256SUMS").is_file() and is_this_release(d):
            return d / "code"
    raise FileNotFoundError(
        f"Could not find the code package of release {RELEASE_ID} (its code/ folder). Download the release "
        f"from the report ('Download the release', in the sidebar or beside each notebook), unzip it, and "
        f"run this notebook from the unzipped folder or its notebooks/ folder. Or set the environment "
        f"variable RELEASE_DIR to the unzipped {RELEASE_ID} folder.")


def find_code_file(relpath):
    """A file inside the release's code package, by its path there (e.g. studies/.../out/result.json)."""
    path = code_dir() / relpath
    if not path.is_file():
        raise FileNotFoundError(f"The code package at {code_dir()} has no {relpath}: is it release {RELEASE_ID}?")
    return path


def load_code_json(relpath):
    """A committed output of the code package, parsed."""
    return json.loads(find_code_file(relpath).read_text(encoding="utf-8"))


def shown(path):
    """A path as printed: relative to this notebook's folder, so an executed notebook names no machine's folders."""
    return os.path.relpath(path, Path.cwd())


def corpus_file(env_var, name):
    """A public corpus file the reader has downloaded: $env_var/name, or name beside this notebook."""
    for d in ([Path(os.environ[env_var])] if os.environ.get(env_var) else []) + [Path.cwd()]:
        if (d / name).is_file():
            return d / name
    raise FileNotFoundError(f"This notebook also needs {name} from the public corpus. Download the corpus "
                            f"(its location and hash are in the release file), then set {env_var} to its "
                            f"folder or put {name} beside this notebook.")


print("Data:", shown(find_release_file()))

Data: per-case-outputs.jsonl


In [2]:
recs = [json.loads(x) for x in open(find_release_file("per-case-outputs.jsonl"), encoding="utf-8")]
floors = [r for r in recs if r["record"] == "floor"]
s4 = lambda x: f"{x:+.4f}".replace("-", "−")
for r in floors:
    lo, hi = r["G_E_minus_G_N_ci95"]
    ok = r["G_N_ci95"][1] <= 0 and lo > 0 and r["G_E_minus_G_N"] >= r["O"] / 2
    assert ("PASS" if ok else "MISS") == r["reading"], r
    assert r["G_N_ci95"][1] < 0
print(len(floors), "floor records; the pass rule reproduces every reading; every G_N interval lies below zero.")

34 floor records; the pass rule reproduces every reading; every G_N interval lies below zero.


In [3]:
SHORT = {"kddcup2010-algebra0607:halfA": "half A", "kddcup2010-bridge06": "bridge06", "edmcup2023": "EDM Cup"}
cell = {(r["model"], SHORT[r["corpus"]], {"quarter": "25%", "fiftieth": "2%"}[r["regime"]]):
        f"{s4(r['G_E_minus_G_N'])} [{s4(r['G_E_minus_G_N_ci95'][0])}, {s4(r['G_E_minus_G_N_ci95'][1])}]" for r in floors}
print(cell[("HawkesKT", "half A", "25%")], cell[("HawkesKT", "bridge06", "25%")], cell[("DKT", "half A", "2%")])
assert cell[("HawkesKT", "half A", "25%")] == "−0.0481 [−0.0566, −0.0370]"
assert cell[("HawkesKT", "bridge06", "25%")] == "−0.0460 [−0.0537, −0.0370]"
assert cell[("DKT", "half A", "2%")] == "−0.0008 [−0.0068, +0.0062]"
assert cell[("AKT", "half A", "25%")] == "+0.0152 [+0.0132, +0.0182]"
misses = sorted(k for k in cell if cell[k].startswith("−"))
assert [r for r in floors if r["reading"] == "MISS"].__len__() == 3
O = {}
for r in floors:
    O.setdefault(SHORT[r["corpus"]], r["O"])
    assert O[SHORT[r["corpus"]]] == r["O"]
DESCRIPTIVE = {"PFA", "BKT", "running-proportion"}
counted = [r for r in floors if r["model"] not in DESCRIPTIVE]
assert {r["model"] for r in counted} == {"DKT", "AKT", "simpleKT", "IEKT", "HawkesKT", "LPKT"}
missed25 = {r["model"] for r in counted if r["regime"] == "quarter" and r["reading"] == "MISS"}
assert missed25 == {"HawkesKT"}       # five of the six models given the floor pass at 25% wherever it was run
print("floors that count: the six deep models';", 6 - len(missed25), "of 6 pass at 25% wherever run")
half = {k: f"{v / 2:.4f}" for k, v in O.items()}
print("bars O/2:", half)
assert half == {"half A": "0.0063", "bridge06": "0.0085", "EDM Cup": "0.0065"}
print("Table 4's checked cells and the bars match the article.")

−0.0481 [−0.0566, −0.0370] −0.0460 [−0.0537, −0.0370] −0.0008 [−0.0068, +0.0062]
floors that count: the six deep models'; 5 of 6 pass at 25% wherever run
bars O/2: {'half A': '0.0063', 'bridge06': '0.0085', 'EDM Cup': '0.0065'}
Table 4's checked cells and the bars match the article.
